# Capstone — Refresh / Content Opportunity Scoring: Which Pages to Fix First?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2) — **Refresh-risk queue**.  
**Question in one line:** *Out of thousands of pages, which ones should a content team refresh first — and why, in words a human trusts?*  
**Output:** a ranked queue (priority score + one reason code + archetype + action) on the 30k starter slice, validated by client-holdout; artifacts and figures feed directly into the deployed paper (`docs/index.html`).

> This notebook mirrors the deployed research paper — same data, same split, same metric (Precision@K), same honest numbers. Run it top-to-bottom (Runtime → Run all).


## 1. Question — the decision, the action, the cost

**What decision does this improve?** *Which pages should an editor fix first this week?* Not "predict decline" as an abstract label — the triage queue.

**Who acts, and what do they do?** A content-ops / editorial team that can review ~50 pages/week opens the ranked queue, checks the **reason code + archetype** on each top-ranked page, verifies position / denominator / query mix, then decides: **Refresh / Investigate / Strategic review / Monitor**. The model *ranks* — the human *decides*.

**What does a wrong answer cost?** False positive (healthy page flagged): 1–3 wasted editor hours + risk of breaking a stable ranking. False negative (missed decline): silent impressions lost. Capacity is scarce, so we optimize **Precision@K at K=50** (and 10/20/100/500), not raw accuracy.

**Why does data/ML help?** Thousands of pages × dozens of tangled signals (staleness, visibility, CTR, position, engagement, age) — no hand-written rule covers the interactions. A learned ranker that weights many signals together beats a fixed `if stale AND moderate then ...` rule. Product flags are outputs, never inputs.

**Framing paragraph (honest):** *For an editorial team deciding which pages to review first, we build a priority score from 90-day search/engagement aggregates, predicting `is_declining = (trend_direction == "down")` measured by Precision@K under a grouped-by-client split. A wrong call costs editor time or missed decay. A plain rule isn't enough because staleness, volume, CTR and position interact non-monotonically and shift per client. We claim only observed / directional / decision-support results.*

**Lane lock:** Refresh / Content Opportunity Scoring. Labels, splits and metrics below are the same as Week-5 model (`training-honest-models`) and Week-6 validation (`hunting-leakage-and-validating`).


In [1]:
# Setup (Colab or local) — runs top-to-bottom with no manual step
import os, sys, pathlib, json, warnings, subprocess
import numpy as np, pandas as pd
import sklearn
warnings.filterwarnings("ignore")
print(f"sklearn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__} | python {sys.version.split()[0]}")
np.random.seed(42)

# Robust path to starter CSV (repo vs Colab)
candidates = [
    "data/raw/content_refresh_anonymized.csv",
    str(pathlib.Path.cwd()/"data/raw/content_refresh_anonymized.csv"),
    "/content/flyrank_intern/data/raw/content_refresh_anonymized.csv",
]
for parent in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
    candidates.append(str(parent/"data/raw/content_refresh_anonymized.csv"))
try:
    root = pathlib.Path(subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
    candidates.append(str(root/"data/raw/content_refresh_anonymized.csv"))
except Exception:
    root = pathlib.Path.cwd()
path = next((c for c in candidates if os.path.exists(c)), None)
if path is None:
    raise FileNotFoundError(f"Missing CSV, tried {candidates}")
print(f"Loading {path}")
df = pd.read_csv(path)
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
for col in ["impressions_90d","clicks_90d","sessions_90d","ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))
print(f"Rows {len(df):,} | clients {df['client_id'].nunique()} | base declining {df['is_declining'].mean():.4f} (n={df['is_declining'].sum():,})")
print(df["trend_direction"].value_counts().to_string())


sklearn 1.9.0 | pandas 3.0.3 | numpy 2.5.1 | python 3.14.7
Loading /home/basel/fly rank assignments/machine_learning/flyrank_intern/data/raw/content_refresh_anonymized.csv
Rows 30,000 | clients 32 | base declining 0.5421 (n=16,262)
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152


## 2. Data — which release, which tables, windows, what was excluded (public-safe)

**Source:** Starter release shipped in this repo: `data/raw/content_refresh_anonymized.csv` — **30,000 rows × 44 columns**, one row per pseudonymized content item, **32 clients**, trailing-90-day aggregates ending at export time. Full column reference: `docs/data-dictionary.md` (keep it open). The production paper notes this is the teaching slice of the ~79M-row warehouse release (`hf://datasets/FlyRank/internship-warehouse`, build v20260703, `2025-01-27→2026-06-30`), and a mid-month warehouse window (`month=2026-03`, feature 03-01→03-15, label 03-16→03-31) was audited for leakage/time design in `work/notebooks/w03_data_contract.ipynb`.

**Date windows:** Single snapshot — no calendar `report_date` here, so honest dimension is **client**, not time. The honest split is grouped by `client_id`. Time-aware validation would require the daily fact; its design is documented and its leakage demo (honest AUC 0.555 → leaky 0.963 when `sh_total_impressions` is added) is in the data-contract notebook.

**What was excluded and why (public-safe):**
- `trend_direction` / `trend_pct` — **never features**; they *define* the label (`is_declining_label = trend_direction=="down"`), using them is leakage (depth-2 tree then splits only on them and hits P@50 1.0 — see `notebooks/02` and Section 5 leakage demo below).
- Pseudonymous IDs (`content_id`, `client_id`) — grouping/joining/splitting only, never features.
- Future/overlapping windows (`*_last_30d`, `*_prev_30d` construction) — they touch the label period; 90-day aggregates are the frozen snapshot but on the warehouse only `*_prev30`-style columns are safe.
- Product flags (`health_score`, `needs_ctr_fix` etc.) not shipped — would be circular (see `flyrank-context` skill).
- `provider_used` / `model_used` not modeled (LLM provenance, not a search signal).

**Gotchas honored:** Rate columns are ×100 percentages (`ctr=0.76` means 0.76%); `avg_position=0` means no data (1,205 rows, 4.0%), not rank zero; `scroll_rate`/`ai_traffic_pct` can exceed 100 (different systems); missingness follows `content_type` (feedly article ~100% missing keyword data, keyword article ~28% missing `word_count`) — median/most-frequent impute with has-flags conceptually, never `fillna(0)`.

**Public-safety:** export contains only hashed IDs + aggregates; no client names, domains, URLs, page titles, keywords or raw queries anywhere (`DATA_USE.md`).


In [2]:
# Verify the Data section with numbers
print(f"avg_position==0 (no data): {(df['avg_position']==0).sum():,} = {(df['avg_position']==0).mean():.1%}")
print(f"scroll_rate>100: {(df['scroll_rate']>100).sum()} | ai_traffic_pct>100: {(df['ai_traffic_pct']>100).sum()} (expected per dictionary)")
print("\nMissingness per content_type:")
miss = df.groupby("content_type", observed=True).agg(
    n=("content_id","count"),
    miss_search=("search_volume", lambda s: s.isna().mean()),
    miss_wc=("word_count", lambda s: s.isna().mean()),
)
print(miss.round(4).to_string())
print("\nTier sizes (need n>=50 per skill):")
for col in ["freshness_tier","impression_tier","position_tier","content_type"]:
    print(f"\n{col}:\n", df[col].value_counts().to_string())


avg_position==0 (no data): 1,205 = 4.0%
scroll_rate>100: 119 | ai_traffic_pct>100: 23 (expected per dictionary)

Missingness per content_type:
                        n  miss_search  miss_wc
content_type                                   
comparison article    697       0.0000    0.000
feedly article       2096       1.0000    0.000
keyword article     27207       0.0137    0.283

Tier sizes (need n>=50 per skill):

freshness_tier:
 freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174

impression_tier:
 impression_tier
low          11248
moderate     10469
good          7205
excellent     1078

position_tier:
 position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319

content_type:
 content_type
keyword article       27207
feedly article         2096
comparison article      697


## 3. Methodology — assumptions, features, label, baseline, validation, leakage checks

**Assumption:** Cross-sectional snapshot — label is an *observed proxy* (bucketed 30-day impression change), not a future prediction. Ranking quality is the claim, not causal recovery.

**Label definition (one sentence):** `is_declining = (trend_direction == "down")` where `down` is `trend_pct < -20%` on `impressions_last_30d vs impressions_prev_30d` (16,262/30,000 = 54.21% positive).

**Features (26 total, `scripts/ml_utils.py`):** 18 numeric: `search_volume, competition, cpc, word_count, char_count, log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d, days_with_impressions, days_with_sessions, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct`; 8 categorical: `competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier`. Never `trend_*` or `*_last_30d/*_prev_30d` or IDs.

**Preprocessing (leakage-safe):** Numerics → `SimpleImputer(median)` (LR adds `StandardScaler`, trees no scaling); Categoricals → `SimpleImputer(most_frequent)` + `OneHotEncoder(handle_unknown="ignore")`. `avg_position==0` kept as 0; rates >100 kept.

**Baseline (transparent rule, beating the hand-written flags):** `score = stale * moderate * impressions_90d` where `stale=(days_since_last_update>=90)` and `moderate=(100<=impressions_90d<3000)`. Reason code: `stale_moderate_visible` if score>0 else `low_priority`; action `refresh` vs `monitor`. Rank by traffic at stake. Full-data P@50 0.74 (mixed clients) — honest comparison uses same grouped holdout.

**Validation design (honest):** `GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)` by `client_id` → 24 clients train (22,885 rows, base 0.550), 8 clients holdout (7,115 rows, base 0.517). The 8 held-out pseudonyms are listed in code. Random `StratifiedShuffleSplit` is shown only as the *before* to expose memorization (RF P@50 0.98 random → 0.70 grouped).

**Leakage checks run below:** forbidden-column audit (10 columns), deliberate `trend_pct` injection (ROC jumps to ~1.0, then removed), ID-not-feature check, timeline diagram, feature-importance sanity (no feature ~0.9).


In [3]:
import sys, pathlib as _pl
for _cand in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent, pathlib.Path(".")]:
    if (_cand/"scripts/ml_utils.py").exists():
        sys.path.insert(0, str(_cand))
        break
try:
    import subprocess as _sp
    _root = pathlib.Path(_sp.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
    sys.path.insert(0, str(_root))
except Exception:
    pass
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

numeric_features = MODEL_NUMERIC_FEATURES
categorical_features = MODEL_CATEGORICAL_FEATURES
print(f"Numeric {len(numeric_features)} | Categorical {len(categorical_features)} = {len(numeric_features+categorical_features)}")
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

# Preprocessing pipelines (fresh instances per model to avoid sharing bug)
def make_lr_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
    ])
def make_rf_preprocess():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
    ])

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    order = np.argsort(-scores)
    k = min(k, len(y_true))
    return float(y_true[order[:k]].mean()) if k>0 else 0.0

y = df["is_declining"]
X = df[numeric_features + categorical_features]
groups = df["client_id"]

# Grouped (honest) + Random (before) splits
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_g, test_g = next(gss.split(X, y, groups=groups))
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_r, test_r = next(sss.split(X, y))
print(f"\nGrouped: train {df.iloc[train_g]['client_id'].nunique()} clients {len(train_g):,} base {y.iloc[train_g].mean():.4f} | test {df.iloc[test_g]['client_id'].nunique()} clients {len(test_g):,} base {y.iloc[test_g].mean():.4f}")
print(f"  Held-out 8: {sorted(df.iloc[test_g]['client_id'].unique().tolist())}")
print(f"Random:  train {df.iloc[train_r]['client_id'].nunique()} clients {len(train_r):,} base {y.iloc[train_r].mean():.4f} | test {df.iloc[test_r]['client_id'].nunique()} clients {len(test_r):,} base {y.iloc[test_r].mean():.4f}")

def build_models():
    lr = Pipeline([("prep", make_lr_preprocess()), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    rf = Pipeline([("prep", make_rf_preprocess()), ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42))])
    tree = Pipeline([("prep", make_rf_preprocess()), ("clf", DecisionTreeClassifier(max_depth=2, random_state=42))])
    return lr, rf, tree

def baseline_scores(dframe):
    return ((dframe["days_since_last_update"]>=90).astype(int) * ((dframe["impressions_90d"]>=100) & (dframe["impressions_90d"]<3000)).astype(int) * dframe["impressions_90d"]).values

def eval_split(name, train_idx, test_idx):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    df_tr, df_te = df.iloc[train_idx], df.iloc[test_idx]
    lr, rf, tree = build_models()
    lr.fit(X_tr, y_tr); rf.fit(X_tr, y_tr); tree.fit(X_tr, y_tr)
    base_te = baseline_scores(df_te)
    rows=[]
    for label, model in [("Baseline (rule)", None), ("LogReg", lr), ("RandomForest", rf), ("Tree d=2", tree)]:
        if label.startswith("Baseline"):
            scores = base_te
            roc = np.nan; ap = np.nan
        else:
            scores = model.predict_proba(X_te)[:,1]
            roc = roc_auc_score(y_te, scores); ap = average_precision_score(y_te, scores)
        rows.append([label, roc, ap] + [precision_at_k(y_te, scores, k) for k in [10,20,50,100,500,1000]])
    # print
    print(f"\n=== {name} ===")
    print(f"{'model':<16} {'ROC':>6} {'PR':>6} {'P@10':>6} {'P@20':>6} {'P@50':>6} {'P@100':>6} {'P@500':>6} {'P@1000':>6}")
    print("-"*78)
    for r in rows:
        roc_s = f"{r[1]:.3f}" if np.isfinite(r[1]) else "  nan"
        ap_s = f"{r[2]:.3f}" if np.isfinite(r[2]) else "  nan"
        print(f"{r[0]:<16} {roc_s:>6} {ap_s:>6} {r[3]:5.3f} {r[4]:5.3f} {r[5]:5.3f} {r[6]:5.3f} {r[7]:5.3f} {r[8]:5.3f}")
    return rows, (lr, rf, tree)

rows_g, models_g = eval_split("Grouped (honest) — client holdout", train_g, test_g)
rows_r, _ = eval_split("Random (before) — rows shuffled", train_r, test_r)
print("\n=== Gap Random - Grouped = memorization bonus ===")
for i, label in enumerate([r[0] for r in rows_g]):
    print(f"{label:<16} ΔP@50 {rows_r[i][5]-rows_g[i][5]:+.3f} (random {rows_r[i][5]:.3f} → grouped {rows_g[i][5]:.3f})")
# Save grouped models for later cells
lr_g, rf_g, tree_g = models_g


Numeric 18 | Categorical 8 = 26
Numeric: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Grouped: train 24 clients 22,885 base 0.5500 | test 8 clients 7,115 base 0.5165
  Held-out 8: ['client_434c9b5ae5', 'client_4e07408562', 'client_8527a891e2', 'client_8b940be7fb', 'client_bdd2d3af3a', 'client_d029fa3a95', 'client_e629fa6598', 'client_f369cb89fc']
Random:  train 32 clients 22,500 base 0.5421 | test 31 clients 7,500 base 0.5420

=== Grouped (honest) — client holdout ===
model               ROC     PR   P@10   P@20   P@50  P@100  P@500 P@1000
---------------------------

## 4. Results — model vs baseline on the same split, with charts

**Honest numbers for the paper (grouped client holdout, test 8 clients n=7,115, base 0.517):** Both learned models beat the transparent rule at every K≥20. The rule collapses under grouping (P@50 0.46, below base) while Logistic Regression P@50 0.72 (36/50) and Random Forest P@50 0.62–0.70 retain signal (ROC ~0.61–0.62, PR ~0.60–0.61). The full-data rule P@50 0.74 is the inflated, mixed-client version — the honest paper number is the grouped one. Random-split RF P@50 0.98 overstates by +0.28 (gap proves memorization).

Charts below are exported for the paper (also in `work/figures/`). Table is the same comparison on the same split.


In [4]:
# Recompute and publish the honest table + figures (P@K curve, archetype risk)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pathlib
import json as _json

# Metrics for export (honest grouped split is the paper number)
ks = [10,20,50,100,200,500,1000]
y_test_g = y.iloc[test_g]
X_test_g = X.iloc[test_g]
df_test_g = df.iloc[test_g]

# Scores on grouped test
rf_scores_g = rf_g.predict_proba(X_test_g)[:,1]
lr_scores_g = lr_g.predict_proba(X_test_g)[:,1]
rule_scores_g = baseline_scores(df_test_g)
base_g = float(y_test_g.mean())

# P@K helper already defined
def p_at_ks(y_true, scores, ks):
    return [precision_at_k(y_true, scores, k) for k in ks]

rf_pk = p_at_ks(y_test_g, rf_scores_g, ks)
lr_pk = p_at_ks(y_test_g, lr_scores_g, ks)
rule_pk = p_at_ks(y_test_g, rule_scores_g, ks)

# Print honest table for the paper
print(f"Base rates: full {y.mean():.4f} | grouped train {y.iloc[train_g].mean():.4f} | grouped test {base_g:.4f}")
print("\nHonest comparison (grouped holdout, test 8 clients, n=7,115):")
hdr = f"{'model':<17} {'ROC':>6} {'PR':>6} " + " ".join([f"P@{k:<4}" for k in ks])
print(hdr)
print("-"*len(hdr))
# ROC/PR from rows_g
for r in rows_g:
    lab, roc, pr = r[0], r[1], r[2]
    pks = r[3:]
    roc_s = f"{roc:.3f}" if np.isfinite(roc) else "  nan"
    pr_s = f"{pr:.3f}" if np.isfinite(pr) else "  nan"
    print(f"{lab:<17} {roc_s:>6} {pr_s:>6} " + " ".join([f"{v:.3f}" for v in pks]))

# --- Figure 1: P@K curve (grouped honest) ---
fig_root = pathlib.Path("work/figures"); fig_root.mkdir(parents=True, exist_ok=True)
import subprocess as _sub
try:
    repo_root = pathlib.Path(_sub.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
except:
    repo_root = pathlib.Path.cwd()

plt.figure(figsize=(10,6))
plt.plot(ks, rf_pk, marker="o", label="RandomForest (grouped)")
plt.plot(ks, lr_pk, marker="s", label="LogReg (grouped)")
plt.plot(ks, rule_pk, marker="^", label="Baseline rule (grouped)")
plt.axhline(base_g, color="gray", linestyle="--", label=f"Base rate {base_g:.3f}")
plt.xscale("log")
plt.xticks(ks, [str(k) for k in ks])
plt.xlabel("K (queue depth, log scale)")
plt.ylabel("Precision@K")
plt.title("Precision@K on grouped client holdout (honest) — base 0.517")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
for p in [fig_root/"playbook_precision_at_k.png", repo_root/"work/outputs/playbook_precision_at_k.png"]:
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p, dpi=150)
plt.close()
print("\nSaved P@K curve to work/figures/playbook_precision_at_k.png")

# --- Feature importance sanity (honest RF, grouped) ---
try:
    ohe = rf_g.named_steps["prep"].named_transformers_["cat"].named_steps["enc"]
    cat_names = list(ohe.get_feature_names_out(categorical_features))
    all_names = numeric_features + cat_names
    imps = rf_g.named_steps["clf"].feature_importances_
    order = np.argsort(imps)[::-1]
    print("\nTop 12 RF importances (honest, grouped):")
    for i in order[:12]:
        print(f"  {all_names[i]:40s} {imps[i]:.4f}")
    print("Sanity: no feature ~0.9 — top ~0.11 is plausible, not label-derived.")
except Exception as e:
    print("importance print skipped:", e)

# --- Save comparison JSON for paper ---
comparison = {
    "base_rate_full": round(float(y.mean()),4),
    "base_rate_test_grouped": round(base_g,4),
    "grouped": {r[0]: {"ROC": None if not np.isfinite(r[1]) else round(float(r[1]),4), "PR": None if not np.isfinite(r[2]) else round(float(r[2]),4),
                        **{f"P@{k}": round(float(v),4) for k,v in zip(ks, r[3:])}} for r in rows_g},
}
out_json = repo_root/"work/outputs/capstone_metrics.json"
out_json.parent.mkdir(parents=True, exist_ok=True)
with open(out_json,"w") as f:
    _json.dump(comparison, f, indent=2)
print(f"\nSaved {out_json}")


Base rates: full 0.5421 | grouped train 0.5500 | grouped test 0.5165

Honest comparison (grouped holdout, test 8 clients, n=7,115):
model                ROC     PR P@10   P@20   P@50   P@100  P@200  P@500  P@1000
--------------------------------------------------------------------------------
Baseline (rule)      nan    nan 0.500 0.450 0.460 0.390 0.494 0.506
LogReg             0.611  0.603 0.800 0.800 0.720 0.660 0.644 0.642
RandomForest       0.615  0.608 0.700 0.700 0.700 0.650 0.646 0.658
Tree d=2           0.577  0.560 0.400 0.550 0.580 0.570 0.584 0.585

Saved P@K curve to work/figures/playbook_precision_at_k.png

Top 12 RF importances (honest, grouped):
  log_impressions_90d                      0.1089
  days_with_impressions                    0.1031
  avg_position                             0.0990
  content_age_days                         0.0834
  char_count                               0.0494
  word_count                               0.0487
  scroll_rate                  

## 5. Limitations & honest framing (observed / directional / decision-support)

**What we can claim:**
- *Observed:* In this 30k-row snapshot (54.2% declining), stale 91–180d pages measured 61.1% declining (n=9,171) vs 51.1% for 0–30d (n=20,480) — a ~10pp lift, same direction in age-split slices.
- *Measured comparison:* On a client-holdout (8 clients, n=7,115, base 0.517), RF P@50 0.70 vs rule 0.46 vs random 0.52 — a directional lift, not 70% magic.
- *Decision-support:* The ranked queue concentrates risk at the top and is a plausible way to spend 50 reviews/week with reason codes + manual verification.

**What we cannot claim (and why):**
- No causation — cross-sectional snapshot, no refresh experiment, no matched design. Cannot say "staleness causes decline" or "refreshing causes recovery" or "predicted Google's algorithm." The honest form: *stale-and-visible pages are empirically more likely to be labeled declining here*.
- Selection bias: staleness was *chosen*, not randomized — part of the gap is the choosing. 181+ stale (n=174) inverts; youngest 31–90 (n=492) declines most (66.9%) — age alone is OPPOSITE.
- No time travel: no calendar `report_date` here; honest dimension is client. Claiming future decline needs a time-aware split on the warehouse daily fact (train past → test future, per-client `gsc_data_start`/`ga4_data_start`, only `*_prev30` safe).
- Small-bucket / denominator warnings: `excellent` n=1,078, `top_3` median 53 impressions — one click swings CTR ~2pp; never automate on `<100` imp or quote a rate without its n. `avg_position==0` is no data, not rank zero.
- Generalization: validated on 8 held-out pseudonymous clients in this slice only; new verticals/templates need re-validation. IDs are grouping only.

**Negative result we keep:** Content age and raw volume are OPPOSITE/MIXED — mechanical oldest/biggest-first refreshing would waste effort.


In [5]:
# Leakage audit — the attack-your-own-model checklist (same as W06)
forbidden = ["trend_pct","trend_direction","is_declining","is_declining_label","impressions_last_30d","impressions_prev_30d","clicks_last_30d","clicks_prev_30d","sessions_last_30d","sessions_prev_30d"]
print("Forbidden in MODEL_NUMERIC_FEATURES?", any(c in MODEL_NUMERIC_FEATURES for c in forbidden))
print("Forbidden in MODEL_CATEGORICAL_FEATURES?", any(c in MODEL_CATEGORICAL_FEATURES for c in forbidden))
for c in forbidden:
    in_num = c in MODEL_NUMERIC_FEATURES; in_cat = c in MODEL_CATEGORICAL_FEATURES
    print(f"  {c:30s} -> {'EXCLUDED ✓' if not (in_num or in_cat) else 'LEAK!'}")
print("\nIDs as features? content_id:", "content_id" in MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES, " client_id:", "client_id" in MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES, "-> grouping only ✓")

# Deliberate leak injection: add trend_pct and watch ROC jump to ~1.0 (then removed)
from sklearn.metrics import roc_auc_score, average_precision_score
X_leak = X.copy()
X_leak["trend_pct"] = df["trend_pct"].fillna(0)
numeric_leak = MODEL_NUMERIC_FEATURES + ["trend_pct"]
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
rf_pre_leak = ColumnTransformer([("num", SimpleImputer(strategy="median"), numeric_leak), ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), MODEL_CATEGORICAL_FEATURES)])
lr_pre_leak = ColumnTransformer([("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_leak), ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), MODEL_CATEGORICAL_FEATURES)])
rf_leak = Pipeline([("prep", rf_pre_leak), ("clf", RandomForestClassifier(n_estimators=100, random_state=42))])
lr_leak = Pipeline([("prep", lr_pre_leak), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
rf_leak.fit(X_leak.iloc[train_g], y.iloc[train_g])
lr_leak.fit(X_leak.iloc[train_g], y.iloc[train_g])
for name, m in [("RF with trend_pct", rf_leak), ("LogReg with trend_pct", lr_leak)]:
    prob = m.predict_proba(X_leak.iloc[test_g])[:,1]
    print(f"LEAKY {name}: ROC {roc_auc_score(y.iloc[test_g], prob):.4f} AP {average_precision_score(y.iloc[test_g], prob):.4f} P@50 {precision_at_k(y.iloc[test_g], prob, 50):.3f} <- near 1.0 is the confession")
print("Leaky feature REMOVED after test — honest numbers above are the ones that count.")

print("\nTimeline: [90d snapshot BEFORE |cutoff| prev_30d (31-60d) vs last_30d (1-30d) = trend_pct -> label is_declining]")
print("Starter 90d aggregates are the frozen snapshot; on warehouse only *_prev30 columns are safe features — *_last30 leaks.")


Forbidden in MODEL_NUMERIC_FEATURES? False
Forbidden in MODEL_CATEGORICAL_FEATURES? False
  trend_pct                      -> EXCLUDED ✓
  trend_direction                -> EXCLUDED ✓
  is_declining                   -> EXCLUDED ✓
  is_declining_label             -> EXCLUDED ✓
  impressions_last_30d           -> EXCLUDED ✓
  impressions_prev_30d           -> EXCLUDED ✓
  clicks_last_30d                -> EXCLUDED ✓
  clicks_prev_30d                -> EXCLUDED ✓
  sessions_last_30d              -> EXCLUDED ✓
  sessions_prev_30d              -> EXCLUDED ✓

IDs as features? content_id: False  client_id: False -> grouping only ✓
LEAKY RF with trend_pct: ROC 1.0000 AP 1.0000 P@50 1.000 <- near 1.0 is the confession
LEAKY LogReg with trend_pct: ROC 0.9986 AP 0.9987 P@50 1.000 <- near 1.0 is the confession
Leaky feature REMOVED after test — honest numbers above are the ones that count.

Timeline: [90d snapshot BEFORE |cutoff| prev_30d (31-60d) vs last_30d (1-30d) = trend_pct -> label is_decli

## 6. Ranked recommendations — the action playbook

**How to read the queue:** Every page gets one **reason code** (mutually exclusive), one **archetype**, one **action**, one **priority**. Primary sort: validated model probability (`rf_prob`); secondary: impressions at stake within a reason code. Reason codes use only observable signals — never `trend_*`.

| Reason code | Human sentence | Rule (observable) | Action | Priority |
|---|---|---|---|---|
| `STALE_MODERATE_AT_RISK` | Stale 91–180d + moderate 100–2,999 + low CTR or page_1/striking | freshness 91–180 & 100≤imp<3000 & (0<ctr≤0.2 or position page_1/striking) | Refresh | P1 |
| `STALE_MODERATE_VISIBLE` | Stale 91–180d + moderate (the baseline) | 91–180 & 100≤imp<3000 | Refresh | P1 |
| `FRESH_BUT_FLAGGED` | Fresh 0–30d but model top-decile | 0–30 & rf_prob ≥ P80 | Investigate | P2 |
| `HIGH_VOLUME_REVIEW` | Good/excellent ≥3,000 | imp≥3000 | Strategic review | P3 |
| `LOW_SIGNAL_MONITOR` | Low <100 | imp<100 | Monitor | P4 |
| `NO_POSITION_DATA` | No GSC data | avg_position==0 | Check instrumentation | P4 |

**Archetypes:** **Fading Performer** (P1, ~11,421) — the queue's core. **Quiet Slider** (P2, ~2,302) — fresh but flagged, model catches what staleness alone misses. **Heavyweight** (P3, 8,283) — ≥3k imp, high cost of error, senior review. **Long Tail** (P4, ~6,789) — <100 imp, monitor/consolidate, not rewrite (38.9% declining — lowest).

**Human-review checklist (every P1/P2 before editing):** 1) position trend + timing, 2) denominator (≥100 imp, days_with_impressions>4), 3) cannibalization/query mix, 4) SERP/seasonality, 5) intent mismatch, 6) duplicate set, 7) log decision.

**No-go list:** Never auto-publish, auto-delete <100, auto-redirect on avg_position 0, bulk-refresh 365+/excellent, apply to new client/warehouse without re-validation, quote ctr without denominator, or claim the queue causes recovery.

Figures for the paper: P@K curve (above) + archetype risk bar (below).


In [6]:
# Build the ranked playbook queue (full 30k scored by grouped-train model) and export artifacts for the paper
import pathlib, subprocess as _sub
try:
    repo_root = pathlib.Path(_sub.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
except:
    repo_root = pathlib.Path.cwd()

# Score FULL data by grouped-train model (simulates deployed ranker); holdout flag retained for honesty
proba_all = rf_g.predict_proba(X)[:,1]
proba_lr_all = lr_g.predict_proba(X)[:,1]
df["rf_prob"] = proba_all
df["lr_prob"] = proba_lr_all
df["baseline_score"] = baseline_scores(df)
df["is_holdout"] = False
df.loc[test_g, "is_holdout"] = True
p80 = float(np.quantile(proba_all, 0.80))
print(f"P80 rf_prob: {p80:.3f} | top-50 mean rf_prob on holdout: {rf_scores_g[np.argsort(-rf_scores_g)[:50]].mean():.3f}")

def assign_playbook(row, p80_thresh):
    imp=row["impressions_90d"]; pos=row["avg_position"]; freshness=row["freshness_tier"]
    if pos==0:
        return "NO_POSITION_DATA","Monitor","Needs-Data-Check",4
    if imp<100:
        return "LOW_SIGNAL_MONITOR","Monitor","Long-Tail",4
    if imp>=3000:
        return "HIGH_VOLUME_REVIEW","Strategic review","Heavyweight",3
    is_stale=(freshness=="91-180")
    is_risky=(0 < row["ctr"] <=0.2) or (row["position_tier"] in ["page_1","striking"])
    if is_stale and is_risky:
        return "STALE_MODERATE_AT_RISK","Refresh","Fading-Performer",1
    if is_stale:
        return "STALE_MODERATE_VISIBLE","Refresh","Fading-Performer",1
    if freshness=="0-30" and row["rf_prob"]>=p80_thresh:
        return "FRESH_BUT_FLAGGED","Investigate","Quiet-Slider",2
    if is_risky:
        return "STALE_MODERATE_AT_RISK","Refresh","Fading-Performer",1
    return "STALE_MODERATE_VISIBLE","Refresh","Fading-Performer",1

play = [assign_playbook(r, p80) for _, r in df.iterrows()]
df[["reason_code","action","archetype","priority"]] = pd.DataFrame(play, index=df.index)

# Archetype summary (observed)
print("\n=== Archetype summary (observed, this snapshot) ===")
for arch, g in df.groupby("archetype", observed=True):
    print(f"{arch:20s} n={len(g):5d} ({len(g)/len(df):.1%}) declining {g['is_declining'].mean():.3f} mean_rf {g['rf_prob'].mean():.3f} median_imp {g['impressions_90d'].median():.0f}")
print("\nReason code summary:")
print(df.groupby("reason_code", observed=True)["is_declining"].agg(mean="mean", n="count", declining="sum").round(4).sort_values("mean", ascending=False).to_string())

# Rank by rf_prob (then impressions at stake)
ranked = df.sort_values(["rf_prob","impressions_90d"], ascending=[False, False]).reset_index(drop=True)
ranked["rank"] = ranked.index+1

# Slice accuracy on honest holdout (for paper's error analysis)
df_test_eval = df.iloc[test_g].copy()
df_test_eval["rf_prob"] = rf_scores_g
df_test_eval["pred_label"] = (rf_scores_g>=0.5).astype(int)
def acc(g): return (g["pred_label"]==g["is_declining"]).mean()
print(f"\nOverall accuracy @0.5 on grouped test: {(df_test_eval['pred_label']==df_test_eval['is_declining']).mean():.3f} (base {y.iloc[test_g].mean():.3f})")
print("Accuracy by freshness_tier (honest):")
for tier,g in df_test_eval.groupby("freshness_tier", observed=True):
    print(f"  {tier:10s} n={len(g):4d} declining {g['is_declining'].mean():.3f} acc {acc(g):.3f}")
# Failure examples
fp = df_test_eval[(df_test_eval["pred_label"]==1) & (df_test_eval["is_declining"]==0)].sort_values("rf_prob", ascending=False)
fn = df_test_eval[(df_test_eval["pred_label"]==0) & (df_test_eval["is_declining"]==1)].sort_values("rf_prob")
cols = ["rf_prob","is_declining","trend_pct","impressions_90d","days_since_last_update","freshness_tier","avg_position","ctr","days_with_impressions","content_age_days","impression_tier","position_tier","reason_code"]
print("\n--- 3 False Positives (flagged but measured not declining) ---")
print(fp[cols].head(3).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\n--- 3 False Negatives (missed declines) ---")
print(fn[cols].head(3).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Export queue for paper + operations
out_cols = ["rank","content_id","client_id","rf_prob","lr_prob","baseline_score","is_declining","reason_code","action","archetype","priority","is_holdout","freshness_tier","impression_tier","position_tier","avg_position","ctr","impressions_90d","days_since_last_update","days_with_impressions","content_age_days","content_type","trend_direction","trend_pct"]
export = ranked[out_cols].copy()
# Write to work/outputs (gitignored) and work/notebooks/work/outputs (for convenience)
for out in [repo_root/"work/outputs/action_playbook_queue.csv", repo_root/"work/notebooks/work/outputs/action_playbook_queue.csv"]:
    out.parent.mkdir(parents=True, exist_ok=True)
    export.to_csv(out, index=False)
print(f"\nWrote queue {len(export):,} rows to work/outputs/action_playbook_queue.csv (holdout flagged)")

# Top-50 / Top-20 for paper tables
for k, name in [(20,"playbook_top20.csv"),(50,"playbook_top50.csv")]:
    topk = ranked.head(k).copy()
    p = repo_root/f"work/outputs/{name}"
    topk.to_csv(p, index=False)
    print(f"Saved {p} ({k} rows, {topk['is_declining'].sum()}/{k} declining in slice)")

# --- Archetype risk figure (observed declining vs base) ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
arch_order = ["Quiet-Slider","Fading-Performer","Heavyweight","Long-Tail","Needs-Data-Check"]
arch_rates = [df[df["archetype"]==a]["is_declining"].mean() if (df["archetype"]==a).any() else 0 for a in arch_order]
arch_ns = [int((df["archetype"]==a).sum()) for a in arch_order]
plt.figure(figsize=(10,6))
bars = plt.bar(arch_order, arch_rates, color=["#6F4E7C","#8BBEE8","#F2C14E","#B8B8B8","#E8A0A0"])
plt.axhline(y.mean(), color="red", linestyle="--", label=f"Base {y.mean():.3f}")
plt.ylabel("Observed declining rate")
plt.title("Archetype risk (observed, this snapshot) — base 0.542")
plt.ylim(0,1)
for bar, rate, n in zip(bars, arch_rates, arch_ns):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f"{rate:.2f}\nn={n}", ha="center", fontsize=9)
plt.legend(); plt.tight_layout()
for p in [repo_root/"work/figures/playbook_archetype_risk.png", repo_root/"work/outputs/playbook_archetype_risk.png"]:
    p.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(p, dpi=150)
plt.close()
print("Saved archetype figure to work/figures/playbook_archetype_risk.png")

# Monitoring snapshot
monitor = {
    "base_full": round(float(y.mean()),4),
    "base_test_grouped": round(float(y.iloc[test_g].mean()),4),
    "avg_position_zero_rate": round(float((df["avg_position"]==0).mean()),4),
    "missing_word_count": round(float(df["word_count"].isna().mean()),4),
    "p80_rf_prob": round(float(p80),4),
    "reason_mix_top200": ranked.head(200)["reason_code"].value_counts().to_dict(),
    "archetype_counts": df["archetype"].value_counts().to_dict(),
}
with open(repo_root/"work/outputs/playbook_monitoring.json","w") as f:
    import json as _js; _js.dump(monitor, f, indent=2)
print("\nMonitoring snapshot:", monitor)


P80 rf_prob: 0.794 | top-50 mean rf_prob on holdout: 0.910

=== Archetype summary (observed, this snapshot) ===
Fading-Performer     n=11421 (38.1%) declining 0.550 mean_rf 0.570 median_imp 729
Heavyweight          n= 8283 (27.6%) declining 0.570 mean_rf 0.585 median_imp 8426
Long-Tail            n= 6789 (22.6%) declining 0.457 mean_rf 0.482 median_imp 17
Needs-Data-Check     n= 1205 (4.0%) declining 0.007 mean_rf 0.018 median_imp 1
Quiet-Slider         n= 2302 (7.7%) declining 0.934 mean_rf 0.860 median_imp 630

Reason code summary:
                          mean     n  declining
reason_code                                    
FRESH_BUT_FLAGGED       0.9340  2302       2150
STALE_MODERATE_AT_RISK  0.6016  7816       4702
HIGH_VOLUME_REVIEW      0.5700  8283       4721
LOW_SIGNAL_MONITOR      0.4569  6789       3102
STALE_MODERATE_VISIBLE  0.4380  3605       1579
NO_POSITION_DATA        0.0066  1205          8

Overall accuracy @0.5 on grouped test: 0.586 (base 0.517)
Accuracy by fresh

## 7. Artifacts the paper embeds

This section is the paper's figure/table factory — every file below is what `docs/index.html` shows.

- **Ranked queue:** `work/outputs/action_playbook_queue.csv` (30,000 rows, scored by grouped-train RF, holdout flagged) — source for any "Top 20/50" table.
- **Top slices:** `work/outputs/playbook_top20.csv`, `playbook_top50.csv`
- **Figures:** `work/figures/playbook_precision_at_k.png` (P@K curve, grouped honest) + `playbook_archetype_risk.png`
- **Metrics:** `work/outputs/capstone_metrics.json` + `work/outputs/playbook_monitoring.json` + `work/outputs/model_metrics.json` (W05) + `work/outputs/signal_audit_evidence.json` (W04)
- **Paper:** `docs/index.html` (GitHub Pages) — links back here.

Signals are ×100 percentages; `avg_position=0` = no data; `scroll_rate`/`ai_traffic_pct` >100 expected; IDs pseudonymous.


In [7]:
# Verify artifacts exist and are public-safe (no raw URLs/names/queries/tokens)
import pathlib, subprocess as _sub, os
try:
    repo_root = pathlib.Path(_sub.check_output(["git","rev-parse","--show-toplevel"], text=True).strip())
except:
    repo_root = pathlib.Path.cwd()
artifacts = [
    "work/outputs/action_playbook_queue.csv",
    "work/outputs/playbook_top20.csv",
    "work/outputs/playbook_top50.csv",
    "work/outputs/capstone_metrics.json",
    "work/outputs/model_metrics.json",
    "work/outputs/signal_audit_evidence.json",
    "work/figures/playbook_precision_at_k.png",
    "work/figures/playbook_archetype_risk.png",
    "work/outputs/playbook_archetype_risk.png",
    "docs/index.html",
]
print("Artifacts for the paper:")
for rel in artifacts:
    p = repo_root/rel
    exists = p.exists()
    size = f"{p.stat().st_size/1024:.1f}KB" if exists else "MISSING"
    print(f"  {'✓' if exists else '✗'} {rel:50s} {size}")
# Public-safety scan: ensure no private-looking columns leaked into exported queue
if (repo_root/"work/outputs/action_playbook_queue.csv").exists():
    import csv as _csv
    with open(repo_root/"work/outputs/action_playbook_queue.csv") as f:
        hdr = next(_csv.reader(f))
    banned = [h for h in hdr if any(s in h.lower() for s in ["url","domain","query","client_name","health_score"])]
    print("\nExport header public-safe?", "YES ✓" if not banned else f"CHECK {banned}")
    print("Header:", hdr[:12], "...")
print("\nPaper URL (if deployed):", (repo_root/"submission/paper_url.txt").read_text().strip() if (repo_root/"submission/paper_url.txt").exists() else "(not yet — next task)")
print("\nCapstone notebook is complete — Runtime → Run all should succeed with no errors.")


Artifacts for the paper:
  ✓ work/outputs/action_playbook_queue.csv             6125.4KB
  ✓ work/outputs/playbook_top20.csv                    8.7KB
  ✓ work/outputs/playbook_top50.csv                    20.7KB
  ✓ work/outputs/capstone_metrics.json                 0.8KB
  ✓ work/outputs/model_metrics.json                    1.9KB
  ✓ work/outputs/signal_audit_evidence.json            2.5KB
  ✓ work/figures/playbook_precision_at_k.png           83.0KB
  ✓ work/figures/playbook_archetype_risk.png           50.4KB
  ✓ work/outputs/playbook_archetype_risk.png           50.4KB
  ✓ docs/index.html                                    31.3KB

Export header public-safe? YES ✓
Header: ['rank', 'content_id', 'client_id', 'rf_prob', 'lr_prob', 'baseline_score', 'is_declining', 'reason_code', 'action', 'archetype', 'priority', 'is_holdout'] ...

Paper URL (if deployed): https://baselsalah342-max.github.io/flyrank_intern/

Capstone notebook is complete — Runtime → Run all should succeed with no err

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all) — seed 42, versions in Cell 1
- [x] No client names, URLs, or private queries anywhere — `content_id`/`client_id` are pseudonyms, aggregated metrics only
- [x] Claims use careful words: *observed, measured, directional, decision-support* — no causal "proves/causes/will recover" or "predicted Google's algorithm"
- [x] Committed to repo under `work/notebooks/` — then submit repo URL on the card. Done.
- [x] Lane locked: Refresh / Content Opportunity Scoring (refresh-risk queue); baseline P@50 0.74 full → 0.46 grouped, model P@50 0.70 (RF) / 0.72 (LR) on same grouped holdout, gap documented
- [x] Leakage audit: forbidden 10 columns excluded ✓, trend_pct injection → ROC 1.0 confession ✓, IDs grouping only ✓, timeline drawn ✓
- [x] Ranked queue exported with reason codes + archetypes + holdout flag; figures regenerated; monitoring snapshot saved for Section 4 triggers
